In [2]:
import pandas as pd

In [3]:
df= pd.read_csv('full_df_cleaned_v3.csv')
df

,Diagnostic,file,target_init,Patient Age,Patient Sex,Target,tarstr,N,D,G,C,A,H,M,O,filename
0,normal fundus,0_right.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",69,Female,"[1, 0, 0, 0, 0, 0, 0, 0]",N,1,0,0,0,0,0,0,0,0_right_69_Female_N.jpg
1,normal fundus,1_right.jpg,"[1, 0, 0, 0, 0, 0, 0, 0]",57,Male,"[1, 0, 0, 0, 0, 0, 0, 0]",N,1,0,0,0,0,0,0,0,1_right_57_Male_N.jpg
2,moderate non proliferative retinopathy,2_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",42,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,2_right_42_Male_D.jpg
3,mild nonproliferative retinopathy,4_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",53,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,4_right_53_Male_D.jpg
4,moderate non proliferative retinopathy,5_right.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",50,Female,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,5_right_50_Female_D.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6108,mild nonproliferative retinopathy,4394_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",56,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,4394_left_56_Male_D.jpg
6109,mild nonproliferative retinopathy,4427_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",43,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,4427_left_43_Male_D.jpg
6110,mild nonproliferative retinopathy,4551_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",53,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,4551_left_53_Male_D.jpg
6111,moderate non proliferative retinopathy,4601_left.jpg,"[0, 1, 0, 0, 0, 0, 0, 0]",36,Male,"[0, 1, 0, 0, 0, 0, 0, 0]",D,0,1,0,0,0,0,0,0,4601_left_36_Male_D.jpg


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import imageio
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import optimizers

In [5]:
df_n = df[df['tarstr']=='N']

In [6]:
df_c = df[df['tarstr']=='C']

In [7]:
df_N_C = pd.concat([df_n, df_c], ignore_index=True)

In [8]:
IMAGE_PATH = '../projectmini/preprocessed_images2/'

In [9]:
df_N_C['filepath'] = IMAGE_PATH + df_N_C['filename']

In [10]:
img_data = []
number_id_nofile = []

for i in range(len(df_N_C)):
  try:
    img_data.append(imageio.imread(df_N_C['filepath'][i]))
  except:
    number_id_nofile.append(df_N_C.index[i])

C:\Users\sruja\AppData\Local\Temp\ipykernel_8512\3197145038.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_data.append(imageio.imread(df_N_C['filepath'][i]))


In [11]:
img_data_array = np.array(img_data)

In [12]:
X = img_data_array

In [13]:
y = df_N_C['C']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, stratify=y)

In [15]:
X_train = X_train / 255
X_test = X_test / 255

In [16]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
model = Sequential([
      Conv2D(32, (3,3), input_shape=(256, 256, 3), activation='relu', padding='same'),
      MaxPooling2D(pool_size=(2,2)),
      Conv2D(64, (3,3), activation='relu', padding='same'),
      MaxPooling2D(pool_size=(2,2)),
      Conv2D(128, (3,3), activation='relu', padding='same'),
      MaxPooling2D(pool_size=(3,3)),
      Conv2D(256, (3,3), activation='relu', padding='same'),
      MaxPooling2D(pool_size=(3,3)),
      Flatten(),
      Dense(120, activation='relu'),
      Dropout(rate=0.5),
      Dense(60, activation='relu'),
      Dropout(rate=0.5),
      Dense(1, activation='sigmoid')])

model.compile(loss='binary_crossentropy', 
        optimizer='adam',
        metrics=['accuracy'])

model.summary()

C:\Users\sruja\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 256, 256, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 128, 128, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 128, 128, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 64, 64, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 64, 64, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 21, 21, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 21, 21, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 7, 7, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 12544)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 120)                 │       1,505,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 120)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 60)                  │           7,260 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 60)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              61 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,901,137 (7.25 MB)

 Trainable params: 1,901,137 (7.25 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def initialize_model():
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(input_shape,)))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [18]:
model.evaluate(X_test, y_test, verbose=0)

[0.7122467756271362, 0.08533916622400284]

In [19]:

import joblib

In [20]:
model.save("baseline_model.keras")

In [21]:
filename = '../projectmini/baseline_model.joblib'
joblib.dump(model, filename)

PicklingError: Can't pickle <function Layer._initialize_tracker.<locals>.<lambda> at 0x0000014E2F0E09A0>: it's not found as keras.src.layers.layer.Layer._initialize_tracker.<locals>.<lambda>